In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

In [ ]:
(X_train, _), (_, _) = keras.datasets.mnist.load_data()
X = X_train.reshape(-1, 784).astype("float32")
X = X / 127.5 - 1.0
print(f"Data shape: {X.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 14s 1us/step
Data shape: (60000, 784)


In [ ]:
def build_generator():
    model = keras.Sequential([
        layers.Dense(128, activation="relu", input_shape=(100,)),
        layers.Dense(256, activation="relu"),
        layers.Dense(512, activation="relu"),
        layers.Dense(784, activation="tanh")
    ])
    return model

def build_discriminator():
    model = keras.Sequential([
        layers.Dense(512, activation="relu", input_shape=(784,)),
        layers.Dense(256, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

In [ ]:
generator = build_generator()
discriminator = build_discriminator()
discriminator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\HD  TECH\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
def train_gan(generator, discriminator, data, epochs=50, batch_size=128):
    np.random.seed(123)
    tf.random.set_seed(123)
    history = {
        "d_loss_real": [],
        "d_acc_real": [],
        "d_loss_fake": [],
        "d_acc_fake": [],
        "g_loss": []
    }

    for epoch in range(epochs):
        idx = np.random.randint(0, data.shape[0], batch_size)
        real_images = data[idx]

        noise = np.random.normal(0, 1, (batch_size, 100))
        fake_images = generator.predict(noise, verbose=0)

        d_loss_real = discriminator.train_on_batch(real_images, np.ones((batch_size, 1)))
        d_loss_fake = discriminator.train_on_batch(fake_images, np.zeros((batch_size, 1)))

        noise = np.random.normal(0, 1, (batch_size, 100))
        discriminator.trainable = False
        fake_images = generator.predict(noise, verbose=0)
        g_loss = discriminator.train_on_batch(fake_images, np.ones((batch_size, 1)))
        discriminator.trainable = True

        history["d_loss_real"].append(float(d_loss_real[0]))
        history["d_acc_real"].append(float(d_loss_real[1]))
        history["d_loss_fake"].append(float(d_loss_fake[0]))
        history["d_acc_fake"].append(float(d_loss_fake[1]))
        history["g_loss"].append(float(g_loss[0]))

        if epoch % 10 == 0:
            print(
                f"Epoch {epoch}: D_loss_real={d_loss_real[0]:.4f}, D_acc_real={d_loss_real[1]:.4f}, ",
                f"D_loss_fake={d_loss_fake[0]:.4f}, D_acc_fake={d_loss_fake[1]:.4f}, G_loss={g_loss[0]:.4f}"
            )

    return history

In [ ]:
history = train_gan(generator, discriminator, X, epochs=50, batch_size=128)
final_accuracy = (history["d_acc_real"][-1] + (1 - history["d_acc_fake"][-1])) / 2

last_k = 10
avg_d_loss_real = float(np.mean(history["d_loss_real"][-last_k:]))
avg_d_loss_fake = float(np.mean(history["d_loss_fake"][-last_k:]))
avg_g_loss = float(np.mean(history["g_loss"][-last_k:]))

eval_size = 1024
idx = np.random.randint(0, X.shape[0], eval_size)
real_eval = X[idx]
noise = np.random.normal(0, 1, (eval_size, 100))
fake_eval = generator.predict(noise, verbose=0)

real_scores = discriminator.predict(real_eval, verbose=0)
fake_scores = discriminator.predict(fake_eval, verbose=0)
y_true = np.vstack([np.ones((eval_size, 1)), np.zeros((eval_size, 1))])
y_scores = np.vstack([real_scores, fake_scores])
y_pred = (y_scores >= 0.5).astype(np.float32)

accuracy = float(np.mean(y_pred == y_true))
tp = float(np.sum((y_pred == 1) & (y_true == 1)))
fp = float(np.sum((y_pred == 1) & (y_true == 0)))
fn = float(np.sum((y_pred == 0) & (y_true == 1)))
precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

auc_metric = tf.keras.metrics.AUC()
auc_metric.update_state(y_true, y_scores)
auc_value = float(auc_metric.result().numpy())

print(f"\nFinal Discriminator Accuracy Score: {final_accuracy:.6f}")
print(f"Average D_loss_real (last {last_k}): {avg_d_loss_real:.6f}")
print(f"Average D_loss_fake (last {last_k}): {avg_d_loss_fake:.6f}")
print(f"Average G_loss (last {last_k}): {avg_g_loss:.6f}")
print(f"Eval Accuracy: {accuracy:.6f}")
print(f"Eval Precision: {precision:.6f}")
print(f"Eval Recall: {recall:.6f}")
print(f"Eval F1: {f1:.6f}")
print(f"Eval AUC: {auc_value:.6f}")

Epoch 0: D_loss_real=1.1507, D_acc_real=0.0078,  D_loss_fake=1.0187, D_acc_fake=0.0039, G_loss=0.9000
Epoch 10: D_loss_real=0.5006, D_acc_real=0.5880,  D_loss_fake=0.5064, D_acc_fake=0.5889, G_loss=0.5129
Epoch 20: D_loss_real=0.4840, D_acc_real=0.6002,  D_loss_fake=0.4876, D_acc_fake=0.5963, G_loss=0.4909
Epoch 30: D_loss_real=0.4782, D_acc_real=0.6030,  D_loss_fake=0.4806, D_acc_fake=0.5994, G_loss=0.4829
Epoch 40: D_loss_real=0.4748, D_acc_real=0.6056,  D_loss_fake=0.4767, D_acc_fake=0.6028, G_loss=0.4785

Final Discriminator Accuracy Score: 0.500258
Average D_loss_real (last 10): 0.473884
Average D_loss_fake (last 10): 0.475521
Average G_loss (last 10): 0.477167
Eval Accuracy: 0.781250
Eval Precision: 0.695652
Eval Recall: 1.000000
Eval F1: 0.820513
Eval AUC: 1.000000
